# Deduplication with Redis Bloom Filters

This notebook demonstrates `DeduplicationService` for preventing duplicate operations using Redis Bloom Filters.

## Why Bloom Filters?

- **Space efficient** - Millions of items in kilobytes
- **O(1) operations** - Constant time add/check
- **No false negatives** - "Not in set" is always correct
- **Configurable false positive rate** - Trade space for accuracy

## Use Cases

- **Duplicate Tool Call Detection** - Prevent re-executing the same tool with same params
- **Cache Stampede Prevention** - Ensure only one process computes a cache miss
- **Request Idempotency** - Prevent duplicate request processing
- **Message Deduplication** - Prevent storing duplicate messages per session

In [ ]:
import os
import asyncio
import uuid

from redis_openai_agents import DeduplicationService

# Configuration
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")

# Initialize DeduplicationService
dedup = DeduplicationService(
    redis_url=REDIS_URL,
    prefix="demo_dedup",
    default_error_rate=0.01,  # 1% false positive rate
)
await dedup.initialize()

print(f"Connected to Redis at {REDIS_URL}")

## 1. Basic Bloom Filter Operations

The fundamental operations: create, add, and check.

In [ ]:
# Create a filter for demo
await dedup.create_filter(
    name="demo_filter",
    capacity=10000,      # Expected number of items
    error_rate=0.01,     # 1% false positive rate
)
print("Created Bloom filter 'demo_filter'")

In [ ]:
# Add some items
items = ["apple", "banana", "cherry"]

for item in items:
    await dedup.add_item("demo_filter", item)
    print(f"Added: {item}")

In [ ]:
# Check for items
test_items = ["apple", "banana", "durian", "elderberry"]

print("Checking items in filter:")
print("=" * 40)
for item in test_items:
    exists = await dedup.check_exists("demo_filter", item)
    status = "FOUND" if exists else "NOT FOUND"
    print(f"  {item}: {status}")

## 2. Duplicate Tool Call Detection

Prevent re-executing the same tool with the same parameters within a time window.

In [ ]:
# Simulate tool calls
tool_calls = [
    ("web_search", {"query": "redis bloom filters"}),
    ("web_search", {"query": "python async"}),
    ("web_search", {"query": "redis bloom filters"}),  # Duplicate!
    ("calculate", {"expression": "2 + 2"}),
    ("calculate", {"expression": "2 + 2"}),  # Duplicate!
]

print("Processing tool calls (5-minute dedup window):")
print("=" * 60)

for tool_name, params in tool_calls:
    is_dup = await dedup.is_duplicate_tool_call(
        tool_name=tool_name,
        params=params,
        window_minutes=5,
    )
    
    status = "DUPLICATE (skipped)" if is_dup else "NEW (executed)"
    print(f"  {tool_name}({params}): {status}")

## 3. Cache Stampede Prevention

When multiple processes have a cache miss simultaneously, only one should compute the result.

In [ ]:
async def simulate_cache_access(process_id: str, query_hash: str) -> str:
    """Simulate a process accessing the cache."""
    
    # Try to acquire lock (only one process should compute)
    should_compute = await dedup.prevent_cache_stampede(
        query_hash=query_hash,
        timeout_seconds=10,
    )
    
    if should_compute:
        # Simulate computing the result
        await asyncio.sleep(0.1)  # Pretend this is an expensive operation
        result = f"Process {process_id} COMPUTED the result"
        
        # Release lock when done
        await dedup.release_cache_lock(query_hash)
    else:
        result = f"Process {process_id} WAITED for result"
    
    return result

# Simulate 5 concurrent cache misses
query_hash = "expensive_query_12345"

print("Simulating concurrent cache misses:")
print("=" * 60)

tasks = [
    simulate_cache_access(f"P{i}", query_hash)
    for i in range(5)
]

results = await asyncio.gather(*tasks)
for result in results:
    print(f"  {result}")

print("\nOnly ONE process computed - others would wait for the cached result.")

## 4. Request Idempotency

Ensure each request is processed exactly once, even with retries.

In [ ]:
# Simulate processing requests with idempotency
requests = [
    "req_001",
    "req_002",
    "req_001",  # Retry of req_001
    "req_003",
    "req_002",  # Retry of req_002
]

print("Processing requests with idempotency:")
print("=" * 60)

for request_id in requests:
    is_new = await dedup.mark_request_processed(request_id)
    
    if is_new:
        status = "PROCESSED (new request)"
    else:
        status = "SKIPPED (duplicate request)"
    
    print(f"  {request_id}: {status}")

## 5. Message Deduplication Per Session

Prevent storing duplicate messages within a session.

In [ ]:
SESSION_ID = "session_demo_123"

# Simulate message stream with duplicates
messages = [
    "Hello, how are you?",
    "I need help with Redis",
    "Hello, how are you?",  # Duplicate!
    "Can you explain Bloom filters?",
    "I need help with Redis",  # Duplicate!
]

print(f"Processing messages for session: {SESSION_ID}")
print("=" * 60)

for msg in messages:
    is_dup = await dedup.is_duplicate_message(SESSION_ID, msg)
    
    if is_dup:
        status = "DUPLICATE (not stored)"
    else:
        status = "NEW (stored)"
    
    print(f"  '{msg[:30]}...': {status}")

## 6. Use Case: Dedup Wrapper for Agent Tools

Create a decorator to automatically deduplicate tool calls.

In [ ]:
from functools import wraps

def deduplicated_tool(window_minutes: int = 5):
    """Decorator to deduplicate tool calls."""
    def decorator(func):
        @wraps(func)
        async def wrapper(*args, **kwargs):
            # Check for duplicate
            is_dup = await dedup.is_duplicate_tool_call(
                tool_name=func.__name__,
                params=kwargs,
                window_minutes=window_minutes,
            )
            
            if is_dup:
                print(f"  [DEDUP] Skipping duplicate call to {func.__name__}")
                return {"cached": True, "message": "Duplicate call skipped"}
            
            return await func(*args, **kwargs)
        return wrapper
    return decorator


@deduplicated_tool(window_minutes=5)
async def expensive_search(query: str) -> dict:
    """An expensive search operation that we want to deduplicate."""
    print(f"  [EXEC] Executing expensive_search(query='{query}')")
    await asyncio.sleep(0.1)  # Simulate work
    return {"results": ["result1", "result2"], "query": query}


# Test the deduplicated tool
print("Testing deduplicated tool:")
print("=" * 60)

print("\nCall 1:")
await expensive_search(query="redis bloom")

print("\nCall 2 (different params):")
await expensive_search(query="python async")

print("\nCall 3 (duplicate of call 1):")
await expensive_search(query="redis bloom")

## Summary

The `DeduplicationService` provides:

1. **Duplicate Tool Call Detection** - Time-windowed deduplication for tool calls
2. **Cache Stampede Prevention** - Distributed locks for cache misses
3. **Request Idempotency** - Ensure each request is processed once
4. **Message Deduplication** - Per-session message dedup

All operations use Redis Bloom Filters for:
- O(1) time complexity
- Minimal memory usage
- Guaranteed no false negatives

In [ ]:
# Cleanup
await dedup.close()
print("Connection closed.")